# Notebook 3: Exploratory Data Analysis (EDA)
**Project:** Car Price Estimation  
**Goal:** Understand distributions, relationships, and patterns in the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

df = pd.read_csv('../data/02_cleaned_data.csv')
print(f'Dataset loaded: {df.shape}')
df.head(3)

## 1. Univariate Analysis
### 1.1 Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Raw price
sns.histplot(df['price'], kde=True, ax=axes[0], color='steelblue', bins=30)
axes[0].set_title('Price Distribution (Raw)', fontsize=14)
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', label=f'Mean: {df["price"].mean():,.0f}')
axes[0].axvline(df['price'].median(), color='orange', linestyle='--', label=f'Median: {df["price"].median():,.0f}')
axes[0].legend()

# Log price
sns.histplot(np.log1p(df['price']), kde=True, ax=axes[1], color='coral', bins=30)
axes[1].set_title('Price Distribution (Log Scale)', fontsize=14)

plt.tight_layout()
plt.savefig('../reports/03_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 Categorical Variables Distribution

In [ ]:
cat_cols = ['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, col in zip(axes.flatten(), cat_cols):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color=sns.color_palette('Set2', len(counts)))
    ax.set_title(f'{col}', fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.3, str(v), ha='center', fontsize=10)

plt.suptitle('Categorical Variables Distribution', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/03_categorical_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.3 Numerical Variables Distribution

In [ ]:
num_cols = ['wheelbase','carlength','carwidth','carheight','curbweight','enginesize',
            'boreratio','stroke','compressionratio','horsepower','peakrpm','citympg','highwaympg']

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue', bins=20)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')

# Turn off empty axes
for ax in axes.flatten()[len(num_cols):]:
    ax.set_visible(False)

plt.suptitle('Numerical Variables Distribution', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../reports/03_numerical_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Bivariate Analysis
### 2.1 Price vs Brand

In [ ]:
brand_price = df.groupby('brand')['price'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(16, 6))
brand_price.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Median Car Price by Brand', fontsize=14)
ax.set_xlabel('Brand')
ax.set_ylabel('Median Price ($)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../reports/03_price_by_brand.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Price vs Fuel Type, Transmission, Door Number

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col in zip(axes, ['fueltype', 'drivewheel', 'carbody']):
    df.boxplot(column='price', by=col, ax=ax, patch_artist=True)
    ax.set_title(f'Price vs {col}', fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('Price ($)')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('')
plt.tight_layout()
plt.savefig('../reports/03_price_vs_categories.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Price vs Engine Size, Horsepower, Curb Weight

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

scatter_pairs = [('enginesize', 'price'), ('horsepower', 'price'), ('curbweight', 'price')]
for ax, (x, y) in zip(axes, scatter_pairs):
    ax.scatter(df[x], df[y], alpha=0.5, color='steelblue')
    m, b, r, p, se = stats.linregress(df[x], df[y])
    x_line = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(x_line, m * x_line + b, color='red', linewidth=2, label=f'r={r:.3f}')
    ax.set_title(f'Price vs {x}', fontsize=12)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend()

plt.tight_layout()
plt.savefig('../reports/03_price_vs_numerics.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Multivariate Analysis
### 3.1 Correlation Matrix

In [ ]:
corr_cols = num_cols + ['price']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with price
price_corr = corr_matrix['price'].drop('price').abs().sort_values(ascending=False)
print('Top correlations with PRICE:')
print(price_corr.to_string())

### 3.2 Pair Plot (Key Features)

In [ ]:
pair_cols = ['price', 'enginesize', 'horsepower', 'curbweight', 'carlength']
pair_df = df[pair_cols + ['fueltype']]

g = sns.pairplot(pair_df, hue='fueltype', diag_kind='kde', plot_kws={'alpha': 0.5})
g.fig.suptitle('Pair Plot: Key Features by Fuel Type', y=1.02, fontsize=14)
plt.savefig('../reports/03_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Key Insights

In [ ]:
print('='*65)
print('KEY INSIGHTS FROM EDA')
print('='*65)

print('\n1. BRAND INFLUENCE:')
luxury_brands = brand_price.head(5).index.tolist()
economy_brands = brand_price.tail(5).index.tolist()
print(f'   Luxury (highest median price): {luxury_brands}')
print(f'   Economy (lowest median price): {economy_brands}')

print('\n2. STRONGEST CORRELATES WITH PRICE:')
for feat, corr in price_corr.head(5).items():
    print(f'   {feat:<25} r = {corr:.3f}')

print('\n3. DISTRIBUTION SKEWNESS:')
for col in ['price', 'enginesize', 'horsepower']:
    skew = df[col].skew()
    print(f'   {col:<20} skewness = {skew:.3f} {"(right-skewed)" if skew > 1 else ""}')

print('\n4. FUEL TYPE:')
ft = df.groupby('fueltype')['price'].median()
print(ft.to_string())

print('\nNotebook 3 Complete!')